# Summary

# Setup

In [1]:
from pprint import pprint
from pathlib import Path
from time import perf_counter
import gc
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import optuna
import torch
from sklearn import set_config

try:
    from catboost import CatBoostClassifier
except ModuleNotFoundError as error:
    if error.name != 'catboost':
        raise
    import subprocess
    import sys
    print('Installing the missing CatBoost package into the active Python environment.', flush=True)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--only-binary=:all:', 'catboost'])
    from catboost import CatBoostClassifier

n_processors = 10
seed = 42
n_trials = 20
timeout_hpo = 7_200     # 2 hours.
n_splits = 5
mlp_max_epochs = 300
mlp_patience = 20
cat_max_iterations = 2_000
cat_patience = 100

device = 'cuda' if torch.cuda.is_available() else 'cpu'
cat_task_type = 'GPU' if device == 'cuda' else 'CPU'
torch.set_num_threads(n_processors)
torch.manual_seed(seed)
np.random.seed(seed)

if device == 'cuda':
    torch.cuda.manual_seed_all(seed)
    print(f'GPU: {torch.cuda.get_device_name(0)}', flush=True)
else:
    print('CUDA is unavailable to PyTorch. Training will use CPU.', flush=True)

set_config(
    transform_output='pandas',
    display='text',
)

RUN_PREVIOUS_STEPS = False
print('Historical model/HPO cells are skipped; Section 5 trains the EDA-improved submission.' if not RUN_PREVIOUS_STEPS else 'Historical model/HPO cells will also run.', flush=True)


c:\Users\yana\Desktop\Library\AI\ai-competitions\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


GPU: NVIDIA GeForce RTX 5090
Historical model/HPO cells are skipped; Section 5 trains the EDA-improved submission.


# 1. Data

## 1) Load

In [2]:
X_train = pd.read_csv('data/training_set_features.csv')
y_train = pd.read_csv('data/training_set_labels.csv')
X_test = pd.read_csv('data/test_set_features.csv')
submission_format = pd.read_csv('data/submission_format.csv')

cols_submit = ['respondent_id', 'h1n1_vaccine', 'seasonal_vaccine']
cols_target = cols_submit[1:]

for frame in [X_train, y_train, X_test, submission_format]:
    if frame['respondent_id'].isna().any() or not frame['respondent_id'].is_unique:
        raise ValueError('respondent_id must be present and unique in every input file.')

if set(X_train['respondent_id']) != set(y_train['respondent_id']):
    raise ValueError('Training feature IDs and label IDs do not match.')

if set(X_test['respondent_id']) != set(submission_format['respondent_id']):
    raise ValueError('Test feature IDs and submission IDs do not match.')

y_train = (
    y_train.set_index('respondent_id')
    .loc[X_train['respondent_id'], cols_target]
    .reset_index(drop=True)
)

if not y_train.isin([0, 1]).all().all():
    raise ValueError('Both target columns must contain only 0 and 1.')

y_train = y_train.astype(np.int32)


def predict_positive_proba(estimator, X):
    probabilities = estimator.predict_proba(X)

    if isinstance(probabilities, list):
        positive = []
        for classes, values in zip(estimator.classes_, probabilities):
            index = np.flatnonzero(np.asarray(classes) == 1)
            positive.append(values[:, index[0]] if len(index) else np.zeros(len(values)))
        probabilities = np.column_stack(positive)

    probabilities = np.asarray(probabilities, dtype=float)
    if probabilities.ndim != 2 or probabilities.shape[0] != X.shape[0]:
        raise ValueError('Expected a probability matrix with one row per sample.')
    if not np.isfinite(probabilities).all():
        raise ValueError('Predicted probabilities contain NaN or infinity.')
    if ((probabilities < 0) | (probabilities > 1)).any():
        raise ValueError('Predicted probabilities must be between 0 and 1.')
    return probabilities


def save_submission(probabilities, filename):
    probabilities = np.asarray(probabilities, dtype=float)
    if probabilities.shape != (len(X_test), len(cols_target)):
        raise ValueError('Expected one probability per test row and target column.')
    if not np.isfinite(probabilities).all():
        raise ValueError('Submission probabilities contain NaN or infinity.')
    if ((probabilities < 0) | (probabilities > 1)).any():
        raise ValueError('Submission probabilities must be between 0 and 1.')

    predictions = pd.DataFrame(
        probabilities,
        index=pd.Index(X_test['respondent_id'], name='respondent_id'),
        columns=cols_target,
    )
    submit = submission_format.loc[:, cols_submit].copy()
    submit[cols_target] = predictions.loc[submit['respondent_id'], cols_target].to_numpy()
    submit.to_csv(filename, index=False)
    print(f'Saved {filename}: {submit.shape}', flush=True)
    return submit


submission_format.head()

,respondent_id,h1n1_vaccine,seasonal_vaccine
0,26707,0.5,0.7
1,26708,0.5,0.7
2,26709,0.5,0.7
3,26710,0.5,0.7
4,26711,0.5,0.7


## 2) Data Type

In [3]:
X_train_num = X_train.select_dtypes(include='number')
X_train_cat = X_train.select_dtypes(exclude='number').astype('category')

cols_num = X_train_num.columns.drop('respondent_id')
cols_cat = X_train_cat.columns

for col in cols_cat:
    print(f"{col:<25s}: {len(X_train_cat[col].value_counts())}")

age_group                : 5
education                : 4
race                     : 4
sex                      : 2
income_poverty           : 3
marital_status           : 2
rent_or_own              : 2
employment_status        : 3
hhs_geo_region           : 10
census_msa               : 3
employment_industry      : 21
employment_occupation    : 23


# 2. Preprocessing

In [4]:
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

prep_num = make_pipeline(
    SimpleImputer(strategy='median'),
    StandardScaler(),
)

prep_cat = make_pipeline(
    SimpleImputer(strategy='most_frequent'),
    OneHotEncoder(handle_unknown='ignore', sparse_output=False),
)

prep = ColumnTransformer(
    transformers=[
        ('prep_num', prep_num, cols_num),
        ('prep_cat', prep_cat, cols_cat),
    ],
).set_output(transform='default')

X_train_prep = prep.fit_transform(X_train)
X_test_prep = prep.transform(X_test)

cols_prep = (
    pd.Index(prep.get_feature_names_out())
    .str.replace('<', 'less_', regex=False)
    .str.replace('>', 'greater_', regex=False)
    .str.replace('[', '', regex=False)
    .str.replace(']', '', regex=False)
)

if not cols_prep.is_unique:
    raise ValueError('Cleaning produced duplicate feature names.')

X_train_prep = pd.DataFrame(X_train_prep, columns=cols_prep, index=X_train.index)
X_test_prep = pd.DataFrame(X_test_prep, columns=cols_prep, index=X_test.index)

print(f'Training features: {X_train_prep.shape} | Test features: {X_test_prep.shape}')

Training features: (26707, 105) | Test features: (26708, 105)


In [5]:
X_train.iloc[0]

respondent_id                                   0
h1n1_concern                                  1.0
h1n1_knowledge                                0.0
behavioral_antiviral_meds                     0.0
behavioral_avoidance                          0.0
behavioral_face_mask                          0.0
behavioral_wash_hands                         0.0
behavioral_large_gatherings                   0.0
behavioral_outside_home                       1.0
behavioral_touch_face                         1.0
doctor_recc_h1n1                              0.0
doctor_recc_seasonal                          0.0
chronic_med_condition                         0.0
child_under_6_months                          0.0
health_worker                                 0.0
health_insurance                              1.0
opinion_h1n1_vacc_effective                   3.0
opinion_h1n1_risk                             1.0
opinion_h1n1_sick_from_vacc                   2.0
opinion_seas_vacc_effective                   2.0


# 3. Base Models

## 1) Baseline

In [6]:
if RUN_PREVIOUS_STEPS:
    from xgboost import XGBClassifier

    base_xgb = XGBClassifier(
        objective='binary:logistic',
        tree_method='hist',
        device=device,
        random_state=seed,
        n_jobs=n_processors,
    )

    print(f'Training baseline XGBoost on {device}', flush=True)
    base_xgb.fit(X_train_prep, y_train)
    base_xgb.target_columns_ = tuple(cols_target)

    y_proba_baseline = predict_positive_proba(base_xgb, X_test_prep)
    submit = save_submission(y_proba_baseline, 'base_xgb.csv')

else:
    print('Skipping completed baseline; proceeding to the EDA-improved model in Section 5.', flush=True)


Skipping completed baseline; proceeding to the EDA-improved model in Section 5.


## 2) XGBoost

In [7]:
if RUN_PREVIOUS_STEPS:
    from sklearn.base import clone
    from sklearn.metrics import roc_auc_score
    from sklearn.model_selection import KFold, train_test_split
    from xgboost.callback import TrainingCallback

    cv = KFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=seed,
    )
    cv_splits = list(cv.split(X_train, y_train))
    cv_data = []

    for fold, (idx_train, idx_valid) in enumerate(cv_splits):
        fold_prep = clone(prep)
        X_fit = fold_prep.fit_transform(X_train.iloc[idx_train])
        X_valid = fold_prep.transform(X_train.iloc[idx_valid])
        cv_data.append({
            'idx_train': idx_train,
            'idx_valid': idx_valid,
            'X_train': np.asarray(X_fit, dtype=np.float32),
            'X_valid': np.asarray(X_valid, dtype=np.float32),
            'y_train': y_train.iloc[idx_train].to_numpy(),
            'y_valid': y_train.iloc[idx_valid].to_numpy(),
        })
        if (y_train.iloc[idx_valid].nunique() != 2).any():
            raise ValueError(f'Validation fold {fold + 1} must contain both classes for each target.')


    def score_auc(estimator, X, y):
        return roc_auc_score(y, predict_positive_proba(estimator, X), average='macro')


    def report_fold(trial, scores, fold, started, name):
        mean_score = float(np.mean(scores))
        trial.report(mean_score, step=fold)
        trial.set_user_attr('fold_scores', list(scores))
        print(
            f'{name} | Trial {trial.number} | Fold {fold + 1}/{n_splits}: '
            f'AUC={scores[-1]:.6f} | Mean={mean_score:.6f} | '
            f'{perf_counter() - started:.1f}s',
            flush=True,
        )
        if fold < n_splits - 1 and trial.should_prune():
            raise optuna.TrialPruned(f'Pruned after {fold + 1} folds.')


    def store_best_oof(cache, trial, score, probabilities):
        if score > cache.get('score', -np.inf):
            cache.update(score=float(score), trial=trial.number, oof=probabilities.copy())


    def run_hpo(objective, name):
        study = optuna.create_study(
            direction='maximize',
            sampler=optuna.samplers.TPESampler(n_startup_trials=5, seed=seed),
            pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=2),
        )
        try:
            study.optimize(
                func=objective,
                n_trials=n_trials,
                timeout=timeout_hpo,
                n_jobs=1,
                gc_after_trial=True,
                show_progress_bar=True,
            )
        except KeyboardInterrupt:
            print(f'{name} HPO interrupted; retaining completed trials.', flush=True)
        finally:
            study.trials_dataframe().to_csv(f'{name}_hpo_trials.csv', index=False)

        if not any(t.state == optuna.trial.TrialState.COMPLETE for t in study.trials):
            raise RuntimeError(f'No {name} trial completed. Finish at least one trial before refitting.')
        print(f'{name} | Best trial: {study.best_trial.number} | CV AUC: {study.best_value:.6f}')
        pprint(study.best_params)
        Path(f'{name}_best_params.json').write_text(json.dumps(study.best_params, indent=2))
        return study


    class ProgressCallback(TrainingCallback):
        def after_iteration(self, model, epoch, evals_log):
            if (epoch + 1) % 200 == 0:
                print(f'Boosting round: {epoch + 1}', flush=True)
            return False


    xgb_cache = {}


    def objective_xgb(trial):
        params = {
            'n_estimators': trial.suggest_int(name='n_estimators', low=300, high=3000, step=300),
            'learning_rate': trial.suggest_float(name='learning_rate', low=0.01, high=0.15, log=True),
            'max_depth': trial.suggest_int(name='max_depth', low=2, high=6),
            'min_child_weight': trial.suggest_float(name='min_child_weight', low=1.0, high=30.0, log=True),
            'subsample': trial.suggest_float(name='subsample', low=0.6, high=1.0),
            'colsample_bytree': trial.suggest_float(name='colsample_bytree', low=0.6, high=1.0),
            'reg_alpha': trial.suggest_categorical(name='reg_alpha', choices=[0.0, 1e-4, 1e-3, 1e-2, 0.1, 1.0, 10.0]),
            'reg_lambda': trial.suggest_float(name='reg_lambda', low=1e-3, high=100.0, log=True),
            'gamma': trial.suggest_categorical(name='gamma', choices=[0.0, 1e-4, 1e-3, 1e-2, 0.05, 0.1]),
        }
        scores = []
        oof = np.full(y_train.shape, np.nan, dtype=float)
        print(f'XGBoost | Trial {trial.number}: {params}', flush=True)

        for fold, data in enumerate(cv_data):
            started = perf_counter()
            print(f'XGBoost | Trial {trial.number} | Fold {fold + 1}/{n_splits}: training', flush=True)
            model = clone(base_xgb).set_params(**params, callbacks=[ProgressCallback()])
            model.fit(data['X_train'], data['y_train'])
            probabilities = predict_positive_proba(model, data['X_valid'])
            oof[data['idx_valid']] = probabilities
            scores.append(float(roc_auc_score(data['y_valid'], probabilities, average='macro')))
            del model
            report_fold(trial, scores, fold, started, 'XGBoost')

        score = float(np.mean(scores))
        store_best_oof(xgb_cache, trial, score, oof)
        return score


    study_xgb = run_hpo(objective_xgb, 'xgb')
    study = study_xgb
    if xgb_cache['trial'] != study_xgb.best_trial.number:
        raise RuntimeError('The stored XGBoost OOF predictions do not match the best completed trial.')
    oof_xgb = xgb_cache['oof']
    np.save('xgb_oof.npy', oof_xgb)

    best_xgb = clone(base_xgb).set_params(**study_xgb.best_params, callbacks=[ProgressCallback()])
    print('Refitting tuned XGBoost on the full training data', flush=True)
    best_xgb.fit(X_train_prep, y_train)
    best_xgb.set_params(callbacks=None)
    best_xgb.target_columns_ = tuple(cols_target)
    best_model = best_xgb

    y_proba_xgb = predict_positive_proba(best_xgb, X_test_prep)
    submit = save_submission(y_proba_xgb, 'base_xgb_hpo.csv')

else:
    print('Skipping completed XGBoost HPO; proceeding to the EDA-improved model in Section 5.', flush=True)


Skipping completed XGBoost HPO; proceeding to the EDA-improved model in Section 5.


## 3) MLP

In [8]:
if RUN_PREVIOUS_STEPS:
    from sklearn.base import BaseEstimator, ClassifierMixin
    from sklearn.utils.validation import check_is_fitted
    from torch import nn


    class TorchMLPClassifier(ClassifierMixin, BaseEstimator):
        def __init__(
            self,
            hidden1=128,
            hidden2=64,
            dropout=0.1,
            learning_rate=1e-3,
            weight_decay=1e-3,
            batch_size=256,
            epochs=300,
            patience=20,
            device='cuda',
            random_state=42,
            verbose=10,
        ):
            self.hidden1 = hidden1
            self.hidden2 = hidden2
            self.dropout = dropout
            self.learning_rate = learning_rate
            self.weight_decay = weight_decay
            self.batch_size = batch_size
            self.epochs = epochs
            self.patience = patience
            self.device = device
            self.random_state = random_state
            self.verbose = verbose

        def fit(self, X, y, eval_set=None):
            X_values = np.asarray(X, dtype=np.float32)
            y_values = np.asarray(y, dtype=np.float32)
            if y_values.ndim != 2 or not np.isin(y_values, [0, 1]).all():
                raise ValueError('MLP targets must be a two-dimensional binary matrix.')
            if not np.isfinite(X_values).all():
                raise ValueError('MLP inputs must be finite after preprocessing.')

            torch.manual_seed(self.random_state)
            if self.device == 'cuda':
                torch.cuda.manual_seed_all(self.random_state)

            self.n_features_in_ = X_values.shape[1]
            self.n_outputs_ = y_values.shape[1]
            self.classes_ = [np.array([0, 1]) for _ in range(self.n_outputs_)]
            self.model_ = nn.Sequential(
                nn.Linear(self.n_features_in_, self.hidden1),
                nn.ReLU(),
                nn.Dropout(self.dropout),
                nn.Linear(self.hidden1, self.hidden2),
                nn.ReLU(),
                nn.Dropout(self.dropout),
                nn.Linear(self.hidden2, self.n_outputs_),
            ).to(self.device)

            X_tensor = torch.as_tensor(X_values, device=self.device)
            y_tensor = torch.as_tensor(y_values, device=self.device)
            optimizer = torch.optim.AdamW(
                self.model_.parameters(),
                lr=self.learning_rate,
                weight_decay=self.weight_decay,
            )
            criterion = nn.BCEWithLogitsLoss()
            self.best_epoch_ = int(self.epochs)
            self.best_score_ = -np.inf
            best_state = None
            patience_score = -np.inf
            patience_epoch = 0
            self.loss_curve_ = []

            if eval_set is not None:
                X_stop, y_stop = eval_set
                X_stop_tensor = torch.as_tensor(np.asarray(X_stop, dtype=np.float32), device=self.device)
                y_stop_values = np.asarray(y_stop)

            for epoch in range(1, self.epochs + 1):
                self.model_.train()
                order = torch.randperm(len(X_tensor), device=self.device)
                total_loss = torch.zeros((), device=self.device)
                for idx in order.split(self.batch_size):
                    optimizer.zero_grad(set_to_none=True)
                    loss = criterion(self.model_(X_tensor[idx]), y_tensor[idx])
                    loss.backward()
                    optimizer.step()
                    total_loss += loss.detach() * len(idx)

                self.n_iter_ = epoch
                loss_value = float((total_loss / len(X_tensor)).item())
                if not np.isfinite(loss_value):
                    raise FloatingPointError('MLP training produced a non-finite loss.')
                self.loss_curve_.append(loss_value)
                message = f'Epoch {epoch}/{self.epochs} | BCE={loss_value:.6f}'

                if eval_set is not None:
                    self.model_.eval()
                    with torch.inference_mode():
                        probabilities = torch.sigmoid(self.model_(X_stop_tensor)).cpu().numpy()
                    score = float(roc_auc_score(y_stop_values, probabilities, average='macro'))
                    message += f' | Stop AUC={score:.6f}'
                    if score > self.best_score_:
                        self.best_score_ = score
                        self.best_epoch_ = epoch
                        best_state = {k: v.detach().clone() for k, v in self.model_.state_dict().items()}
                    if score > patience_score + 1e-4:
                        patience_score = score
                        patience_epoch = epoch

                stop = eval_set is not None and epoch - patience_epoch >= self.patience
                if self.verbose and (epoch == 1 or epoch % self.verbose == 0 or stop or epoch == self.epochs):
                    print(message, flush=True)
                if stop:
                    break

            if best_state is not None:
                self.model_.load_state_dict(best_state)
            self.model_.eval().to('cpu')
            return self

        def predict_proba(self, X):
            check_is_fitted(self, 'model_')
            values = np.asarray(X, dtype=np.float32)
            if values.shape[1] != self.n_features_in_:
                raise ValueError('MLP input feature count does not match training.')
            self.model_.to(self.device).eval()
            outputs = []
            try:
                with torch.inference_mode():
                    for start in range(0, len(values), 4096):
                        batch = torch.as_tensor(values[start:start + 4096], device=self.device)
                        outputs.append(torch.sigmoid(self.model_(batch)).cpu().numpy())
            finally:
                self.model_.to('cpu')
            return np.concatenate(outputs, axis=0)

        def predict(self, X):
            return (self.predict_proba(X) >= 0.5).astype(np.int32)


    inner_splits = []
    mlp_stop_data = []

    for data in cv_data:
        idx_train = data['idx_train']
        strata = y_train.iloc[idx_train, 0] * 2 + y_train.iloc[idx_train, 1]
        stratify = strata if strata.value_counts().min() >= 2 else None
        idx_fit, idx_stop = train_test_split(
            idx_train,
            test_size=0.15,
            random_state=seed,
            stratify=stratify,
        )
        if (y_train.iloc[idx_stop].nunique() != 2).any():
            raise ValueError('The MLP early-stopping set must contain both classes per target.')
        inner_splits.append((idx_fit, idx_stop))
        stop_prep = clone(prep)
        mlp_stop_data.append({
            'X_train': np.asarray(stop_prep.fit_transform(X_train.iloc[idx_fit]), dtype=np.float32),
            'X_stop': np.asarray(stop_prep.transform(X_train.iloc[idx_stop]), dtype=np.float32),
            'y_train': y_train.iloc[idx_fit].to_numpy(),
            'y_stop': y_train.iloc[idx_stop].to_numpy(),
        })

    mlp_architectures = {
        '64_32': (64, 32),
        '128_64': (128, 64),
        '256_64': (256, 64),
        '256_128': (256, 128),
    }
    mlp_cache = {}


    def make_mlp(params, epochs, random_state=seed):
        hidden1, hidden2 = mlp_architectures[params['architecture']]
        return TorchMLPClassifier(
            hidden1=hidden1,
            hidden2=hidden2,
            dropout=params['dropout'],
            learning_rate=params['learning_rate'],
            weight_decay=params['weight_decay'],
            batch_size=params['batch_size'],
            epochs=epochs,
            patience=mlp_patience,
            device=device,
            random_state=random_state,
            verbose=10,
        )


    def objective_mlp(trial):
        params = {
            'architecture': trial.suggest_categorical('architecture', list(mlp_architectures)),
            'learning_rate': trial.suggest_float('learning_rate', 1e-4, 3e-3, log=True),
            'weight_decay': trial.suggest_float('weight_decay', 1e-5, 1e-1, log=True),
            'dropout': trial.suggest_float('dropout', 0.0, 0.3),
            'batch_size': trial.suggest_categorical('batch_size', [128, 256, 512]),
        }
        scores = []
        best_epochs = []
        oof = np.full(y_train.shape, np.nan, dtype=float)
        print(f'MLP | Trial {trial.number}: {params}', flush=True)

        for fold, (data, stop_data) in enumerate(zip(cv_data, mlp_stop_data)):
            started = perf_counter()
            print(f'MLP | Trial {trial.number} | Fold {fold + 1}/{n_splits}: selecting epochs', flush=True)
            pilot = make_mlp(params, mlp_max_epochs, seed + fold)
            pilot.fit(
                stop_data['X_train'],
                stop_data['y_train'],
                eval_set=(stop_data['X_stop'], stop_data['y_stop']),
            )
            epochs = int(pilot.best_epoch_)
            best_epochs.append(epochs)
            del pilot

            print(f'MLP | Fold {fold + 1}: refitting the training fold for {epochs} epochs', flush=True)
            model = make_mlp(params, epochs, seed + fold)
            model.fit(data['X_train'], data['y_train'])
            probabilities = predict_positive_proba(model, data['X_valid'])
            oof[data['idx_valid']] = probabilities
            scores.append(float(roc_auc_score(data['y_valid'], probabilities, average='macro')))
            trial.set_user_attr('best_epochs', best_epochs.copy())
            del model
            report_fold(trial, scores, fold, started, 'MLP')

        score = float(np.mean(scores))
        store_best_oof(mlp_cache, trial, score, oof)
        return score


    study_mlp = run_hpo(objective_mlp, 'mlp')
    if mlp_cache['trial'] != study_mlp.best_trial.number:
        raise RuntimeError('The stored MLP OOF predictions do not match the best completed trial.')
    oof_mlp = mlp_cache['oof']
    np.save('mlp_oof.npy', oof_mlp)

    mlp_refit_epochs = max(1, int(np.median(study_mlp.best_trial.user_attrs['best_epochs'])))
    print(f'Refitting tuned MLP on the full training data for {mlp_refit_epochs} epochs', flush=True)
    best_mlp = make_mlp(study_mlp.best_params, mlp_refit_epochs)
    best_mlp.fit(X_train_prep, y_train)
    best_mlp.target_columns_ = tuple(cols_target)
    base_mlp = best_mlp

    y_proba_mlp = predict_positive_proba(best_mlp, X_test_prep)
    submit = save_submission(y_proba_mlp, 'base_mlp_hpo.csv')
    Path('mlp_refit_epochs.json').write_text(json.dumps({'epochs': mlp_refit_epochs}, indent=2))
    gc.collect()
    if device == 'cuda':
        torch.cuda.empty_cache()

else:
    print('Skipping completed MLP HPO; proceeding to the EDA-improved model in Section 5.', flush=True)


Skipping completed MLP HPO; proceeding to the EDA-improved model in Section 5.


## 4) CatBoost

In [9]:
if RUN_PREVIOUS_STEPS:
    class CatBoostMultiLabelClassifier(ClassifierMixin, BaseEstimator):
        def __init__(self, params=None, iterations=2000, cat_features=None, verbose=200):
            self.params = params
            self.iterations = iterations
            self.cat_features = cat_features
            self.verbose = verbose

        def _prepare(self, X):
            frame = X.loc[:, self.feature_names_in_].copy()
            for col in self.cat_features:
                frame[col] = frame[col].astype(object).where(frame[col].notna(), '__MISSING__').astype(str)
            return frame

        def fit(self, X, y, eval_set=None):
            self.feature_names_in_ = np.asarray([c for c in X.columns if c != 'respondent_id'], dtype=object)
            self.n_features_in_ = len(X.columns)
            self.target_columns_ = tuple(y.columns)
            self.classes_ = [np.array([0, 1]) for _ in self.target_columns_]
            X_values = self._prepare(X)
            X_stop = self._prepare(eval_set[0]) if eval_set is not None else None
            counts = [int(self.iterations)] * len(self.target_columns_) if np.isscalar(self.iterations) else list(self.iterations)
            if len(counts) != len(self.target_columns_):
                raise ValueError('CatBoost needs one iteration count per target.')
            self.estimators_ = []
            self.best_iterations_ = []

            for index, col in enumerate(self.target_columns_):
                print(f'CatBoost | {col} | {cat_task_type} | Maximum iterations: {counts[index]}', flush=True)
                fixed = {
                    'iterations': int(counts[index]),
                    'loss_function': 'Logloss',
                    'eval_metric': 'AUC',
                    'task_type': cat_task_type,
                    'thread_count': n_processors,
                    'random_seed': seed,
                    'bootstrap_type': 'Bayesian',
                    'boosting_type': 'Plain',
                    'border_count': 128,
                    'allow_writing_files': False,
                    'cat_features': self.cat_features,
                }
                if cat_task_type == 'GPU':
                    fixed.update(devices='0', gpu_ram_part=0.7)
                model = CatBoostClassifier(**fixed, **(self.params or {}))
                if eval_set is None:
                    model.fit(X_values, y[col], verbose=self.verbose)
                else:
                    model.fit(
                        X_values,
                        y[col],
                        eval_set=(X_stop, eval_set[1][col]),
                        early_stopping_rounds=cat_patience,
                        use_best_model=True,
                        verbose=self.verbose,
                    )
                self.estimators_.append(model)
                self.best_iterations_.append(int(model.tree_count_))
            return self

        def predict_proba(self, X):
            check_is_fitted(self, 'estimators_')
            X_values = self._prepare(X)
            return np.column_stack([
                model.predict_proba(X_values, thread_count=n_processors)[:, 1]
                for model in self.estimators_
            ])

        def predict(self, X):
            return (self.predict_proba(X) >= 0.5).astype(np.int32)


    cat_cache = {}


    def objective_catboost(trial):
        params = {
            'depth': trial.suggest_int('depth', 4, 7),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
            'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1.0, 30.0, log=True),
            'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 2.0),
        }
        scores = []
        best_iterations = []
        oof = np.full(y_train.shape, np.nan, dtype=float)
        print(f'CatBoost | Trial {trial.number}: {params}', flush=True)

        for fold, ((idx_train, idx_valid), (idx_fit, idx_stop)) in enumerate(zip(cv_splits, inner_splits)):
            started = perf_counter()
            print(f'CatBoost | Trial {trial.number} | Fold {fold + 1}/{n_splits}: selecting iterations', flush=True)
            pilot = CatBoostMultiLabelClassifier(
                params=params,
                iterations=cat_max_iterations,
                cat_features=cols_cat.tolist(),
            )
            pilot.fit(
                X_train.iloc[idx_fit],
                y_train.iloc[idx_fit],
                eval_set=(X_train.iloc[idx_stop], y_train.iloc[idx_stop]),
            )
            iterations = pilot.best_iterations_.copy()
            best_iterations.append(iterations)
            del pilot

            print(f'CatBoost | Fold {fold + 1}: refitting the training fold for {iterations} iterations', flush=True)
            model = CatBoostMultiLabelClassifier(
                params=params,
                iterations=iterations,
                cat_features=cols_cat.tolist(),
            )
            model.fit(X_train.iloc[idx_train], y_train.iloc[idx_train])
            probabilities = predict_positive_proba(model, X_train.iloc[idx_valid])
            oof[idx_valid] = probabilities
            scores.append(float(roc_auc_score(y_train.iloc[idx_valid], probabilities, average='macro')))
            trial.set_user_attr('best_iterations', best_iterations.copy())
            del model
            report_fold(trial, scores, fold, started, 'CatBoost')

        score = float(np.mean(scores))
        store_best_oof(cat_cache, trial, score, oof)
        return score


    study_catboost = run_hpo(objective_catboost, 'catboost')
    if cat_cache['trial'] != study_catboost.best_trial.number:
        raise RuntimeError('The stored CatBoost OOF predictions do not match the best completed trial.')
    oof_catboost = cat_cache['oof']
    np.save('catboost_oof.npy', oof_catboost)

    cat_refit_iterations = np.maximum(
        1,
        np.median(study_catboost.best_trial.user_attrs['best_iterations'], axis=0).astype(int),
    ).tolist()
    print(f'Refitting tuned CatBoost on the full training data for {cat_refit_iterations} iterations', flush=True)
    best_catboost = CatBoostMultiLabelClassifier(
        params=study_catboost.best_params,
        iterations=cat_refit_iterations,
        cat_features=cols_cat.tolist(),
    )
    best_catboost.fit(X_train, y_train)
    base_catboost = best_catboost

    y_proba_catboost = predict_positive_proba(best_catboost, X_test)
    submit = save_submission(y_proba_catboost, 'base_catboost_hpo.csv')
    _ = Path('catboost_refit_iterations.json').write_text(json.dumps(dict(zip(cols_target, cat_refit_iterations)), indent=2))

else:
    print('Skipping completed CatBoost HPO; proceeding to the EDA-improved model in Section 5.', flush=True)


Skipping completed CatBoost HPO; proceeding to the EDA-improved model in Section 5.


# 4. Ensemble 

## 1) Voting - Soft

In [10]:
if RUN_PREVIOUS_STEPS:
    class PrefitSoftVotingClassifier(ClassifierMixin, BaseEstimator):
        def __init__(self, estimators, weights=None):
            self.estimators = estimators
            self.weights = weights

        def fit(self, X, y):
            self.target_columns_ = tuple(y.columns)
            self.n_features_in_ = X.shape[1]
            self.n_outputs_ = y.shape[1]
            self.classes_ = [np.array([0, 1]) for _ in self.target_columns_]
            self.weights_ = np.asarray(
                self.weights if self.weights is not None else np.ones(len(self.estimators)),
                dtype=float,
            )
            if (
                self.weights_.shape != (len(self.estimators),)
                or not np.isfinite(self.weights_).all()
                or (self.weights_ < 0).any()
                or self.weights_.sum() <= 0
            ):
                raise ValueError('Use one finite nonnegative weight per model, with a positive sum.')
            self.estimators_ = []
            for name, model in self.estimators:
                check_is_fitted(model)
                if model.target_columns_ != self.target_columns_:
                    raise ValueError(f'Target order mismatch in {name}.')
                self.estimators_.append(model)
            self.named_estimators_ = dict(self.estimators)
            return self

        def predict_proba(self, X):
            check_is_fitted(self, 'estimators_')
            active = [(model, weight) for model, weight in zip(self.estimators_, self.weights_) if weight > 0]
            probabilities = [predict_positive_proba(model, X) for model, _ in active]
            if any(p.shape != (len(X), self.n_outputs_) for p in probabilities):
                raise ValueError('Each model must return one positive-class probability per target.')
            return np.average(np.stack(probabilities), axis=0, weights=[weight for _, weight in active])

        def predict(self, X):
            return (self.predict_proba(X) >= 0.5).astype(np.int32)


    oof_predictions = {
        'xgb': oof_xgb,
        'catboost': oof_catboost,
        'mlp': oof_mlp,
    }
    for name, probabilities in oof_predictions.items():
        if probabilities.shape != y_train.shape or not np.isfinite(probabilities).all():
            raise ValueError(f'{name} does not have complete, aligned OOF predictions.')

    candidates = {
        'xgb': [1, 0, 0],
        'catboost': [0, 1, 0],
        'mlp': [0, 0, 1],
        'xgb_catboost': [1, 1, 0],
        'xgb_catboost_mlp': [1, 1, 1],
    }
    oof_stack = np.stack(list(oof_predictions.values()))
    rows = []
    for name, weights in candidates.items():
        probabilities = np.average(oof_stack, axis=0, weights=weights)
        scores = np.asarray([
            roc_auc_score(y_train.iloc[idx_valid], probabilities[idx_valid], average=None)
            for _, idx_valid in cv_splits
        ])
        rows.append({
            'model': name,
            'mean_cv_auc': float(scores.mean()),
            'h1n1_auc': float(scores[:, 0].mean()),
            'seasonal_auc': float(scores[:, 1].mean()),
            'std_cv_auc': float(scores.mean(axis=1).std()),
        })

    cv_results = pd.DataFrame(rows).set_index('model')
    print(cv_results.sort_values('mean_cv_auc', ascending=False).to_string(float_format=lambda v: f'{v:.6f}'))
    cv_results.to_csv('cv_results.csv')
    print('These CV scores were used for model selection; they are not an independent test score.', flush=True)

    ensemble_names = ['xgb_catboost', 'xgb_catboost_mlp']
    selected_ensemble = cv_results.loc[ensemble_names, 'mean_cv_auc'].idxmax()
    selected_weights = candidates[selected_ensemble]
    print(f'Selected ensemble: {selected_ensemble} | Weights: {selected_weights}', flush=True)

    xgb_pipeline = make_pipeline(prep, best_xgb)
    mlp_pipeline = make_pipeline(prep, best_mlp)
    xgb_pipeline.target_columns_ = tuple(cols_target)
    mlp_pipeline.target_columns_ = tuple(cols_target)

    base_models = [
        ('xgb', xgb_pipeline),
        ('catboost', best_catboost),
        ('mlp', mlp_pipeline),
    ]
    voting_soft = PrefitSoftVotingClassifier(estimators=base_models, weights=selected_weights)
    voting_soft.fit(X_train, y_train)

    print('Predicting with the selected soft-voting ensemble; no base-model retraining', flush=True)
    y_proba = voting_soft.predict_proba(X_test)
    y_pred = (y_proba >= 0.5).astype(np.int32)

    submit = save_submission(y_proba, 'voting_soft.csv')
    submit.to_csv('submission.csv', index=False)
    Path('ensemble_selection.json').write_text(json.dumps({
        'selected_ensemble': selected_ensemble,
        'model_order': ['xgb', 'catboost', 'mlp'],
        'weights': selected_weights,
        'mean_cv_auc': float(cv_results.loc[selected_ensemble, 'mean_cv_auc']),
    }, indent=2))
    print('Submission ready: submission.csv', flush=True)
    submit.head()

else:
    print('Skipping completed previous ensemble; proceeding to the EDA-improved model in Section 5.', flush=True)


Skipping completed previous ensemble; proceeding to the EDA-improved model in Section 5.


# 5. EDA

In [11]:
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.model_selection import KFold
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.utils.validation import check_is_fitted
from xgboost import XGBClassifier
from importlib.metadata import version
from io import BytesIO
import hashlib
import zipfile
import joblib

eda_snapshot = {'targets': ['h1n1_vaccine', 'seasonal_vaccine'], 'seed': 42, 'n_splits': 5, 'xgb_params': {'n_estimators': 900, 'learning_rate': 0.012152146607279167, 'max_depth': 6, 'min_child_weight': 9.657382120888048, 'subsample': 0.6868127153463741, 'colsample_bytree': 0.9284331680796784, 'reg_alpha': 0.1, 'reg_lambda': 0.8588607193321974, 'gamma': 0.0}, 'data_sha256': {'data/training_set_features.csv': '9a805f24b45fd9bd0b52cedf7cab0ccc62b512372f00151b028b004b0973d1e5', 'data/training_set_labels.csv': '0ad33a821b96666a0dbfafc4e6fe23969ded212c6b55eb8763bd8bd78b8cae40', 'data/test_set_features.csv': 'fbb8e4f208c222a1559c701dd351e5db8fc12c1671393a009bdc055830fa42b5', 'data/submission_format.csv': '9a6cfd8e7d75f4d0586f59ec50d27947c5c2d546e25db80bce29df53ebee3a35'}, 'catboost_oof_sha256': 'ba4fbfbccfcb1073bbb467764cc6b43fa1f9682e3433d6b1b8451bf3f51ba8ad', 'catboost_submission_sha256': 'c7697c0fe69d2420745e5138f0013db6a30d795e29d50a50a23dbf06f5b9c7e1', 'aligned_oof_sha256': '11aa7a17d2f6ba35ddd97abd772074ef6853213cec377bd4ca996cdd4b72eacd', 'catboost_candidate_sha256': '4d4e348d57377770235e5e3faa371d3b5e0f5d1d2bbf54a575f5c69f886edcfc', 'catboost_mean_fold_auc': [0.8716695662050264, 0.8647888649705442]}

eda_started = perf_counter()
eda_output = Path('main_eda_assets')
eda_output.mkdir(exist_ok=True)
eda_cache = eda_output / 'cv_cache'
eda_cache.mkdir(exist_ok=True)
eda_targets = list(cols_target)
eda_feature_columns = [c for c in X_train.columns if c != 'respondent_id']
eda_versions = {name: version(name) for name in ['numpy', 'pandas', 'scikit-learn', 'xgboost']}

if eda_targets != eda_snapshot['targets'] or (seed, n_splits) != (eda_snapshot['seed'], eda_snapshot['n_splits']):
    raise ValueError('Keep the target order, seed and fold count used by the completed EDA.')

for filename, expected in eda_snapshot['data_sha256'].items():
    path = Path(filename)
    if not path.is_file() or hashlib.sha256(path.read_bytes()).hexdigest() != expected:
        raise ValueError(f'{filename} differs from the completed EDA data. Do not reuse its saved OOF predictions with changed data.')


def eda_result_bytes(name):
    path = Path('eda_results') / name
    if path.is_file():
        return path.read_bytes()
    path = Path('eda_results.zip')
    if path.is_file():
        with zipfile.ZipFile(path) as archive:
            if name in archive.namelist():
                return archive.read(name)
    return None


def eda_probability_check(values, rows, label):
    values = np.asarray(values, dtype=float)
    if values.shape != (rows, 2) or not np.isfinite(values).all():
        raise ValueError(f'{label}: expected a finite ({rows}, 2) probability matrix.')
    if ((values < 0) | (values > 1)).any():
        raise ValueError(f'{label}: probabilities must be in [0, 1].')
    return values


eda_ids = X_train['respondent_id'].to_numpy()
eda_test_ids = X_test['respondent_id'].to_numpy()
eda_y_values = y_train.loc[:, eda_targets].to_numpy(dtype=np.int32)
eda_splits = list(KFold(n_splits=n_splits, shuffle=True, random_state=seed).split(X_train))
for fold, (tr, va) in enumerate(eda_splits):
    if any(np.unique(eda_y_values[idx, j]).size != 2 for idx in [tr, va] for j in range(2)):
        raise ValueError(f'Fold {fold + 1} needs both classes of each target in training and validation.')

eda_saved_oof = eda_result_bytes('02_aligned_oof_predictions.csv')
if eda_saved_oof is not None:
    if hashlib.sha256(eda_saved_oof).hexdigest() != eda_snapshot['aligned_oof_sha256']:
        raise ValueError('The EDA OOF table differs from the reviewed run; do not mix result bundles.')
    eda_saved = pd.read_csv(BytesIO(eda_saved_oof))
    if eda_saved.respondent_id.duplicated().any() or set(eda_saved.respondent_id) != set(eda_ids):
        raise ValueError('EDA OOF respondent IDs do not match training data.')
    eda_saved = eda_saved.set_index('respondent_id').loc[eda_ids]
    if not np.array_equal(eda_saved[eda_targets].to_numpy(), eda_y_values):
        raise ValueError('EDA OOF target values do not match the current labels.')
    oof_cat_eda = eda_saved[[f'catboost__{t}' for t in eda_targets]].to_numpy()
else:
    path = Path('catboost_oof.npy')
    if not path.is_file():
        raise FileNotFoundError('Keep eda_results.zip, eda_results/, or catboost_oof.npy beside this notebook.')
    if hashlib.sha256(path.read_bytes()).hexdigest() != eda_snapshot['catboost_oof_sha256']:
        raise ValueError('catboost_oof.npy is from a different run.')
    oof_cat_eda = np.load(path, allow_pickle=False)
oof_cat_eda = eda_probability_check(oof_cat_eda, len(X_train), 'CatBoost OOF')

eda_saved_test = eda_result_bytes('candidates/catboost_only.csv')
if eda_saved_test is not None:
    if hashlib.sha256(eda_saved_test).hexdigest() != eda_snapshot['catboost_candidate_sha256']:
        raise ValueError('The CatBoost candidate file differs from the reviewed EDA run.')
    eda_saved = pd.read_csv(BytesIO(eda_saved_test))
else:
    path = Path('base_catboost_hpo.csv')
    if not path.is_file() or hashlib.sha256(path.read_bytes()).hexdigest() != eda_snapshot['catboost_submission_sha256']:
        raise ValueError('Keep the original base_catboost_hpo.csv, or the reviewed eda_results.zip, in this folder.')
    eda_saved = pd.read_csv(path)
if eda_saved.respondent_id.duplicated().any() or set(eda_saved.respondent_id) != set(eda_test_ids):
    raise ValueError('Saved CatBoost test IDs do not match X_test.')
y_proba_cat_eda = eda_probability_check(
    eda_saved.set_index('respondent_id').loc[eda_test_ids, eda_targets].to_numpy(),
    len(X_test), 'CatBoost test predictions',
)


def eda_fold_auc(probability):
    return np.array([
        [roc_auc_score(eda_y_values[va, j], probability[va, j]) for j in range(2)]
        for _, va in eda_splits
    ])


eda_cat_scores = eda_fold_auc(oof_cat_eda)
if not np.allclose(eda_cat_scores.mean(axis=0), eda_snapshot['catboost_mean_fold_auc'], rtol=0, atol=2e-6):
    raise ValueError('The saved CatBoost fold AUCs do not reproduce the reviewed EDA scores.')
print('Verified original data and CatBoost predictions. Reusing CatBoost; no CatBoost/MLP training or HPO.', flush=True)


class EDAFeaturePreprocessor(TransformerMixin, BaseEstimator):
    def __init__(self, numeric_imputation='native'):
        self.numeric_imputation = numeric_imputation

    def _parts(self, X):
        frame = X.loc[:, self.feature_names_in_].copy()
        flags = frame[self.numeric_columns_].isna().astype(np.float32).add_suffix('__missing')
        if 'employment_status' in frame:
            status = frame['employment_status'].astype('string').str.strip().str.casefold()
            tags = pd.Series('__MISSING__OTHER_OR_UNKNOWN_STATUS', index=frame.index, dtype=object)
            for value, token in {
                'not in labor force': '__MISSING__NOT_IN_LABOR_FORCE',
                'unemployed': '__MISSING__UNEMPLOYED',
                'employed': '__MISSING__EMPLOYED',
            }.items():
                tags.loc[status.eq(value).fillna(False)] = token
            for col in ['employment_industry', 'employment_occupation']:
                if col in frame:
                    observed = frame[col].notna()
                    original = frame.loc[observed, col].astype(object)
                    frame[col] = frame[col].astype(object)
                    frame.loc[~observed, col] = tags.loc[~observed]
                    if not frame.loc[observed, col].equals(original):
                        raise AssertionError('An observed employment answer was changed.')
        for col in self.opinion_columns_:
            flags[f'{col}__unknown'] = frame[col].eq(3).astype(np.float32)
            flags[f'{col}__missing'] = frame[col].isna().astype(np.float32)
            frame[col] = frame[col].map({1: 0., 2: 1., 4: 2., 5: 3.})
        numeric = frame[self.numeric_columns_].to_numpy(dtype=float)
        if np.isinf(numeric).any():
            raise ValueError('Numeric input contains infinity.')
        categorical = frame[self.categorical_columns_].astype(object)
        categorical = categorical.where(categorical.notna(), np.nan)
        return numeric, categorical, flags.to_numpy(dtype=np.float32)

    def fit(self, X, y=None):
        if self.numeric_imputation not in ['median', 'native']:
            raise ValueError('Use median or native numeric missing-value handling.')
        self.feature_names_in_ = np.asarray([c for c in X.columns if c != 'respondent_id'], dtype=object)
        self.n_features_in_ = len(self.feature_names_in_)
        features = X.loc[:, self.feature_names_in_]
        self.numeric_columns_ = features.select_dtypes('number').columns.tolist()
        self.categorical_columns_ = features.select_dtypes(exclude='number').columns.tolist()
        self.opinion_columns_ = [c for c in self.numeric_columns_ if c.startswith('opinion_')]
        numeric, categorical, _ = self._parts(X)
        self.numeric_transformer_ = (
            make_pipeline(SimpleImputer(strategy='median', keep_empty_features=True), StandardScaler())
            if self.numeric_imputation == 'median' else StandardScaler()
        )
        self.numeric_transformer_.fit(numeric)
        self.categorical_transformer_ = None
        if self.categorical_columns_:
            self.categorical_transformer_ = make_pipeline(
                SimpleImputer(strategy='constant', fill_value='__MISSING__', keep_empty_features=True),
                OneHotEncoder(handle_unknown='ignore', sparse_output=False),
            ).set_output(transform='default')
            self.categorical_transformer_.fit(categorical)
        return self

    def transform(self, X):
        check_is_fitted(self, 'numeric_transformer_')
        numeric, categorical, flags = self._parts(X)
        numeric = self.numeric_transformer_.transform(numeric)
        categorical = (
            self.categorical_transformer_.transform(categorical)
            if self.categorical_transformer_ is not None else np.empty((len(X), 0))
        )
        result = np.ascontiguousarray(np.column_stack([numeric, categorical, flags]), dtype=np.float32)
        if np.isinf(result).any() or (self.numeric_imputation == 'median' and np.isnan(result).any()):
            raise ValueError('Preprocessing produced invalid numeric values.')
        return result


def eda_new_xgb(mode):
    return make_pipeline(
        EDAFeaturePreprocessor(numeric_imputation=mode),
        XGBClassifier(
            **eda_snapshot['xgb_params'],
            objective='binary:logistic', tree_method='hist', device=device,
            n_jobs=n_processors, random_state=seed,
        ),
    )


eda_oof = {'catboost': oof_cat_eda}
eda_cv_rows = []
for mode in ['median', 'native']:
    name = f'xgb_{mode}'
    probability = np.full((len(X_train), 2), np.nan)
    for fold, (tr, va) in enumerate(eda_splits):
        started = perf_counter()
        descriptor = {
            'revision': 1, 'mode': mode, 'fold': fold, 'seed': seed, 'n_splits': n_splits,
            'data': eda_snapshot['data_sha256'], 'params': eda_snapshot['xgb_params'],
            'versions': eda_versions, 'device': device, 'n_jobs': n_processors,
        }
        key = hashlib.sha256(json.dumps(descriptor, sort_keys=True).encode()).hexdigest()[:24]
        cache_path = eda_cache / f'{key}.npz'
        cached = cache_path.is_file()
        print(f'EDA model | {name} | Fold {fold + 1}/{n_splits}: ' + ('loading' if cached else 'training'), flush=True)
        if cached:
            with np.load(cache_path, allow_pickle=False) as saved:
                if not np.array_equal(saved['respondent_id'], eda_ids[va]):
                    raise ValueError('Cached fold row IDs do not match.')
                fold_probability = eda_probability_check(saved['probability'], len(va), name)
        else:
            pipeline = eda_new_xgb(mode)
            pipeline.fit(X_train.iloc[tr], y_train.iloc[tr])
            pipeline[-1].set_params(device='cpu')
            fold_probability = eda_probability_check(pipeline.predict_proba(X_train.iloc[va]), len(va), name)
            np.savez_compressed(cache_path, respondent_id=eda_ids[va], probability=fold_probability)
            del pipeline
        probability[va] = fold_probability
        scores = [roc_auc_score(eda_y_values[va, j], fold_probability[:, j]) for j in range(2)]
        eda_cv_rows.append({
            'model': name, 'fold': fold + 1, 'macro_auc': float(np.mean(scores)),
            'h1n1_auc': scores[0], 'seasonal_auc': scores[1],
            'seconds': perf_counter() - started, 'cached': cached,
        })
        print(f'{name} | Fold {fold + 1}: AUC={np.mean(scores):.6f} | {perf_counter() - started:.1f}s', flush=True)
    eda_oof[name] = eda_probability_check(probability, len(X_train), name)

eda_candidates = {'catboost_only': oof_cat_eda}
eda_candidate_specs = {'catboost_only': {'xgb_mode': None, 'xgb_weight': 0.}}
for mode in ['median', 'native']:
    for weight in [.25, .5, .75, 1.]:
        name = f'xgb_{mode}_{int(100 * weight)}pct'
        eda_candidates[name] = weight * eda_oof[f'xgb_{mode}'] + (1 - weight) * oof_cat_eda
        eda_candidate_specs[name] = {'xgb_mode': mode, 'xgb_weight': weight}

eda_candidate_scores = {name: eda_fold_auc(p) for name, p in eda_candidates.items()}
eda_table = pd.DataFrame([
    {'candidate': name, 'macro_auc': scores.mean(), 'h1n1_auc': scores[:, 0].mean(),
     'seasonal_auc': scores[:, 1].mean(), 'fold_sd': scores.mean(axis=1).std(ddof=1)}
    for name, scores in eda_candidate_scores.items()
]).set_index('candidate')
print(eda_table.sort_values('macro_auc', ascending=False).to_string(float_format=lambda x: f'{x:.6f}'))
eda_selected = {}
for j, target in enumerate(eda_targets):
    name = max(eda_candidate_scores, key=lambda c: float(eda_candidate_scores[c][:, j].mean()))
    eda_selected[target] = dict(candidate=name, **eda_candidate_specs[name])
    print(f'{target}: {name} | mean fold AUC={eda_candidate_scores[name][:, j].mean():.6f}', flush=True)
eda_selected_oof = np.column_stack([
    eda_candidates[eda_selected[target]['candidate']][:, j] for j, target in enumerate(eda_targets)
])
eda_selected_score = float(eda_fold_auc(eda_selected_oof).mean())
print(f'Selected model-selection CV AUC: {eda_selected_score:.6f}. This is not an independent test score.', flush=True)

models_eda = {}
y_proba_models_eda = {'catboost': y_proba_cat_eda}
for mode in ['median', 'native']:
    print(f'Refitting EDA XGBoost ({mode}) on all training rows; no early-stopping holdout.', flush=True)
    pipeline = eda_new_xgb(mode)
    pipeline.fit(X_train, y_train)
    pipeline[-1].set_params(device='cpu')
    models_eda[f'xgb_{mode}'] = pipeline
    y_proba_models_eda[f'xgb_{mode}'] = eda_probability_check(
        pipeline.predict_proba(X_test), len(X_test), f'xgb_{mode} test',
    )
    pipeline[-1].save_model(str(eda_output / f'xgb_{mode}.ubj'))
    joblib.dump(pipeline[0].__dict__, eda_output / f'xgb_{mode}_preprocessing.joblib')
    save_submission(y_proba_models_eda[f'xgb_{mode}'], str(eda_output / f'xgb_{mode}.csv'))


def predict_eda_from_components(component_probabilities):
    result = np.empty((len(component_probabilities['catboost']), 2))
    for j, target in enumerate(eda_targets):
        selection = eda_selected[target]
        weight = selection['xgb_weight']
        result[:, j] = (1 - weight) * component_probabilities['catboost'][:, j]
        if weight > 0:
            result[:, j] += weight * component_probabilities[f'xgb_{selection["xgb_mode"]}'][:, j]
    return result


y_proba_eda = eda_probability_check(predict_eda_from_components(y_proba_models_eda), len(X_test), 'EDA submission')
submit_eda = save_submission(y_proba_eda, 'submission_eda.csv')
eda_table.to_csv(eda_output / 'candidate_cv.csv')
pd.DataFrame(eda_cv_rows).to_csv(eda_output / 'xgb_fold_scores.csv', index=False)
eda_fold_table = pd.DataFrame([
    {'candidate': name, 'fold': fold + 1, 'target': target, 'auc': float(scores[fold, j])}
    for name, scores in eda_candidate_scores.items()
    for fold in range(n_splits) for j, target in enumerate(eda_targets)
])
eda_fold_table.to_csv(eda_output / 'candidate_fold_scores.csv', index=False)
eda_oof_export = pd.DataFrame({'respondent_id': eda_ids})
for j, target in enumerate(eda_targets):
    eda_oof_export[target] = eda_y_values[:, j]
    for name, probability in {**eda_oof, 'selected': eda_selected_oof}.items():
        eda_oof_export[f'{name}__{target}'] = probability[:, j]
eda_oof_export.to_csv(eda_output / 'aligned_oof_predictions.csv', index=False)
eda_run = {
    'selected': eda_selected, 'mean_cv_auc': eda_selected_score,
    'snapshot': eda_snapshot, 'versions': eda_versions,
    'elapsed_seconds': perf_counter() - eda_started,
    'catboost_reused': True, 'original_hpo_rerun': RUN_PREVIOUS_STEPS,
    'evaluation': 'Five original HPO folds; preprocessing and voting selected on CV, not independent validation.',
    'submission': 'submission_eda.csv',
}
(eda_output / 'run.json').write_text(json.dumps(eda_run, indent=2), encoding='utf-8')
with zipfile.ZipFile('main_eda_results.zip', 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for name in ['candidate_cv.csv', 'xgb_fold_scores.csv', 'candidate_fold_scores.csv', 'aligned_oof_predictions.csv', 'run.json']:
        archive.write(eda_output / name, arcname=name)
print(f'Finished in {(perf_counter() - eda_started) / 60:.1f} minutes. Submit submission_eda.csv.', flush=True)
print('Original submission.csv and HPO assets were not changed by Section 5. Diagnostics: main_eda_results.zip.', flush=True)
submit_eda.head()


Verified original data and CatBoost predictions. Reusing CatBoost; no CatBoost/MLP training or HPO.
EDA model | xgb_median | Fold 1/5: training
xgb_median | Fold 1: AUC=0.870080 | 4.7s
EDA model | xgb_median | Fold 2/5: training
xgb_median | Fold 2: AUC=0.866671 | 4.7s
EDA model | xgb_median | Fold 3/5: training
xgb_median | Fold 3: AUC=0.867906 | 4.6s
EDA model | xgb_median | Fold 4/5: training
xgb_median | Fold 4: AUC=0.867046 | 4.6s
EDA model | xgb_median | Fold 5/5: training
xgb_median | Fold 5: AUC=0.868058 | 4.9s
EDA model | xgb_native | Fold 1/5: training
xgb_native | Fold 1: AUC=0.870854 | 4.7s
EDA model | xgb_native | Fold 2/5: training
xgb_native | Fold 2: AUC=0.867247 | 4.7s
EDA model | xgb_native | Fold 3/5: training
xgb_native | Fold 3: AUC=0.867399 | 4.7s
EDA model | xgb_native | Fold 4/5: training
xgb_native | Fold 4: AUC=0.867155 | 4.9s
EDA model | xgb_native | Fold 5/5: training
xgb_native | Fold 5: AUC=0.868335 | 4.7s
                   macro_auc  h1n1_auc  seasonal_a

,respondent_id,h1n1_vaccine,seasonal_vaccine
0,26707,0.128401,0.215009
1,26708,0.030821,0.031474
2,26709,0.198437,0.709683
3,26710,0.652223,0.895098
4,26711,0.313028,0.470117
